# Week 02 - AutoFix

We will try to build a Harness that deals with code remediation, a key challenge in modern DevOps.

The “AutoFix” system should automatically propose fixes for issues identified by a language server, and apply them to increase code quality.

## Goal: Automated Issue Resolution
Our primary task is to implement an `auto_fix` function that serves as the core of this system. This function will act as a bridge between **diagnostic output** and **code changes**, using an LLM to generate a precise solution.

**Inputs**:
- A codebase path
- An error object returned by the language server (e.g. a type-checking failure).

**Output**:
- A proposed change in unified diff format to resolve the issue.

Focus on designing an effective LLM prompt and context strategy that reliably converts a structured error message into a correct and syntactically valid unified diff.

## Environment Variables & Imports

Before using this notebook, please ensure you have obtained an API Key from your LLM backend and update the `.env` file to include it as follows:

```bash
GOOGLE_API_KEY=<copy API key here>
OPENAI_API_KEY=<copy API key here>
ANTRHOPIC_API_KEY=<copy API key here>
LLAMA_CPP_API_KEY=<copy API key here>
```

In [ ]:
import initialize_notebook # noqa

In [ ]:
import pathlib
from typing import Any

import jinja2
from IPython import display as ipydisplay
from IPython.display import Code

from hslu.dlm03.common import backend as backend_lib
from hslu.dlm03.common import presets
from hslu.dlm03.tools import lint, unified_diff
from hslu.dlm03.util import file as file_lib

## Parameters

In [ ]:
backend_name = "Gemma 4 26B"
backend_config = presets.CONFIGS_BY_NAME[backend_name]
backend = backend_lib.LLM.from_config(backend_config)

In [ ]:
code_folder = pathlib.Path("/Users/vincent/Development/Test/")

## Code reading

This part implements helpers for reading files from the given code base.

In [ ]:
Code("../util/file.py")

In [ ]:
files = file_lib.Files.read_from(code_folder, "**/*.py")
for file in files.files:
    print("==========", file.file_path, "==========")
    print(file.content)
    print()

## Linter

This section computes linting issues on the loaded codebase.

In [ ]:
Code("../tools/lint.py")

In [ ]:
issues = lint.lint(code_folder, relative=True)
for issue in issues.issues:
    print(f"{issue.filename}:{issue.location.row}:{issue.location.column} {issue.code} {issue.message}")

## AutoFix Implementation

In [ ]:
system_instructions_template_string = \
"""You are an expert Software Engineer tasked to solve issues found in code.
You will be provided with some code files, and a list of issues found by a linter,
and you should output the fixed code such that the issues are not present anymore."""
system_instructions_template = jinja2.Template(system_instructions_template_string, undefined=jinja2.StrictUndefined)

In [ ]:
user_message_template_string = \
"""Files:
{% for file in files.files %}
# {{ file.file_path }}
{% for line in file.content.splitlines(keepends=True) %}
{{- loop.index }}\t{{ line -}}
{% endfor %}
{% endfor %}

Issues:
{%- for issue in issues.issues %}
{{ issue.filename }}:{{ issue.location.row }}:{{ issue.location.column }} {{ issue.code }} {{ issue.message }}
{%- endfor -%}
"""
user_message_template = jinja2.Template(user_message_template_string, undefined=jinja2.StrictUndefined)

In [ ]:
user_message = user_message_template.render(files=files, issues=issues)
system_instructions = system_instructions_template.render(files=files, issues=issues)
print("========== System Instructions ==========")
print(system_instructions)
print("========== User Message ==========")
print(user_message)

In [ ]:
def auto_fix(
    issues: lint.Issues,
    files: file_lib.Files,
    backend: backend_lib.LLM,
    *,
    system_instructions_template: jinja2.Template,
    user_message_template: jinja2.Template,
    **kwargs: Any,
) -> list[unified_diff.UnifiedDiff]:
    system_instructions = system_instructions_template.render(iles=files, issues=issues, **kwargs)
    user_message = user_message_template.render(files=files, issues=issues, **kwargs)
    history = [
        {"role": "system", "content": system_instructions},
        {"role": "user", "content": user_message},
    ]
    response = backend(response_format=file_lib.Files, messages=history)
    new_files = response.choices[0].message.parsed
    return files.diff(new_files)

In [ ]:
fixes = auto_fix(
    issues,
    files,
    backend,
    system_instructions_template=system_instructions_template,
    user_message_template=user_message_template,
)

In [ ]:
for fix in fixes:
    ipydisplay.display(ipydisplay.HTML(fix.to_html()))

## YOLOFix

In [ ]:
reverse_stack = []
retries = 5

def yolo_fix(
    issues: lint.Issues,
    files: file_lib.File,
    backend: backend_lib.LLM,
    *,
    system_instructions_template: jinja2.Template,
    user_message_template: jinja2.Template,
    keep: int | None = None,
    retries: int = 0,
    **kwargs: Any,
) -> list[unified_diff.UnifiedDiff]:
    tries = 0
    while tries <= retries:
        try:
            if keep:
                issues = lint.Issues(issues=issues.issues[:keep])
            fixes = auto_fix(
                issues,
                files,
                backend,
                system_instructions_template=system_instructions_template,
                user_message_template=user_message_template,
                **kwargs,
            )
            for fix in fixes:
                unified_diff.apply(fix, cwd=code_folder)
        except Exception:
            tries += 1
            if tries > retries:
                raise
        else:
            return fixes
    return []

while issues := lint.lint(code_folder, relative=True):
    files = file_lib.Files.read_from(code_folder, "**/*.py")
    print(f"Found {len(issues.issues)} issues remaining.")
    if len(issues.issues) == 0:
        break
    fixes = yolo_fix(issues, files, backend, system_instructions_template=system_instructions_template,
            user_message_template=user_message_template, keep=None, retries=retries)
    reverse_stack.extend(fix.revert() for fix in fixes)
print("Done!")